# 🎯 Multi-Object Detection & Tracking Pipeline - Demo Notebook

This notebook demonstrates the CV tracking pipeline with interactive controls for video analysis.

## Setup

In [ ]:
import sys
sys.path.insert(0, '..')

import cv2
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets
from pathlib import Path
import json

from src.pipeline import TrackingPipeline, PipelineConfig
from src.detector import Detector
from src.tracker import MultiTracker
from src.annotator import VideoAnnotator
from src.analytics import HeatmapGenerator, TeamClusterer

print("✅ All modules loaded successfully!")

## 1. Load and Process Video

In [ ]:
# Configuration
VIDEO_PATH = "../temp/sample_video.mp4"  # Update with your video path
OUTPUT_DIR = "../outputs/demo"

# Create pipeline with custom config
config = PipelineConfig(
    detection_model="yolov8m.pt",
    confidence_threshold=0.35,
    primary_tracker="kalman",
    reid_enabled=True,
    enable_heatmap=True,
    enable_team_clustering=True,
    frame_skip=2
)

pipeline = TrackingPipeline(config=config)
print(f"Pipeline initialized with config:")
print(f"  - Detection model: {config.detection_model}")
print(f"  - Confidence: {config.confidence_threshold}")
print(f"  - Tracker: {config.primary_tracker}")
print(f"  - ReID enabled: {config.reid_enabled}")

In [ ]:
# Progress callback for notebook
progress_widget = widgets.FloatProgress(
    value=0,
    min=0,
    max=1.0,
    description='Processing:',
    bar_style='info',
    style={'bar_color': '#667eea'},
    layout=widgets.Layout(width='100%')
)

status_label = widgets.Label(value='Initializing...')

def update_progress(progress, message):
    progress_widget.value = progress
    status_label.value = message

display(widgets.VBox([progress_widget, status_label]))

In [ ]:
# Run pipeline (uncomment to process)
# results = pipeline.run(VIDEO_PATH, progress_callback=update_progress)
# print(f"\n✅ Processing complete!")
# print(f"Output directory: {results['output_dir']}")

## 2. Interactive Frame Viewer

In [ ]:
class FrameViewer:
    """Interactive frame viewer with scrubbing."""
    
    def __init__(self, video_path):
        self.cap = cv2.VideoCapture(video_path)
        self.total_frames = int(self.cap.get(cv2.CAP_PROP_FRAME_COUNT))
        self.fps = self.cap.get(cv2.CAP_PROP_FPS)
        self.current_frame = 0
        
        # Create widgets
        self.slider = widgets.IntSlider(
            value=0,
            min=0,
            max=self.total_frames - 1,
            description='Frame:',
            continuous_update=False,
            layout=widgets.Layout(width='80%')
        )
        
        self.play_button = widgets.Button(
            description='▶ Play',
            button_style='success'
        )
        
        self.output = widgets.Output()
        
        # Connect callbacks
        self.slider.observe(self._on_slider_change, names='value')
        
    def _on_slider_change(self, change):
        self.show_frame(change['new'])
    
    def show_frame(self, frame_idx):
        self.cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ret, frame = self.cap.read()
        
        if ret:
            with self.output:
                clear_output(wait=True)
                
                # Convert BGR to RGB
                frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                
                fig, ax = plt.subplots(figsize=(12, 8))
                ax.imshow(frame_rgb)
                ax.set_title(f'Frame {frame_idx} / {self.total_frames} | Time: {frame_idx/self.fps:.2f}s')
                ax.axis('off')
                plt.tight_layout()
                plt.show()
    
    def display(self):
        self.show_frame(0)
        display(widgets.VBox([
            self.slider,
            self.output
        ]))
    
    def close(self):
        self.cap.release()

# Usage (uncomment with valid video):
# viewer = FrameViewer("../outputs/demo/output_annotated.mp4")
# viewer.display()

## 3. Visualize Results

In [ ]:
def visualize_results(output_dir):
    """Visualize all analytics outputs."""
    output_path = Path(output_dir)
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # Heatmap
    heatmap_path = output_path / "heatmap.png"
    if heatmap_path.exists():
        img = plt.imread(str(heatmap_path))
        axes[0].imshow(img)
        axes[0].set_title('Movement Heatmap')
    axes[0].axis('off')
    
    # Count plot
    count_path = output_path / "count_plot.png"
    if count_path.exists():
        img = plt.imread(str(count_path))
        axes[1].imshow(img)
        axes[1].set_title('Object Count Over Time')
    axes[1].axis('off')
    
    # Sample frame
    sample_path = output_path / "sample_frame_0.jpg"
    if sample_path.exists():
        img = plt.imread(str(sample_path))
        axes[2].imshow(img)
        axes[2].set_title('Sample Annotated Frame')
    axes[2].axis('off')
    
    plt.tight_layout()
    plt.show()

# visualize_results("../outputs/demo")

## 4. Metrics Analysis

In [ ]:
def analyze_metrics(output_dir):
    """Load and display tracking metrics."""
    metrics_path = Path(output_dir) / "metrics.json"
    
    if not metrics_path.exists():
        print("Metrics file not found")
        return
    
    with open(metrics_path) as f:
        metrics = json.load(f)
    
    print("=" * 50)
    print("         TRACKING METRICS")
    print("=" * 50)
    
    for key, value in metrics.items():
        if isinstance(value, float):
            print(f"{key:30s}: {value:.3f}")
        else:
            print(f"{key:30s}: {value}")
    
    print("=" * 50)
    
    # Visualize key metrics
    fig, ax = plt.subplots(figsize=(10, 6))
    
    metrics_to_plot = {
        'Total Tracks': metrics.get('total_tracks', 0),
        'Avg Track Length': metrics.get('avg_track_length', 0),
        'ID Switches': metrics.get('id_switches', 0),
        'Fragmentations': metrics.get('fragmentations', 0)
    }
    
    bars = ax.bar(metrics_to_plot.keys(), metrics_to_plot.values(), color=['#667eea', '#764ba2', '#f39c12', '#e74c3c'])
    ax.set_ylabel('Count')
    ax.set_title('Tracking Performance Metrics')
    
    for bar, val in zip(bars, metrics_to_plot.values()):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, 
                f'{val:.1f}', ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    plt.show()

# analyze_metrics("../outputs/demo")

## 5. Single Frame Detection Demo

In [ ]:
def demo_detection(image_path_or_array):
    """Run detection on a single image."""
    # Load image
    if isinstance(image_path_or_array, str):
        frame = cv2.imread(image_path_or_array)
    else:
        frame = image_path_or_array
    
    # Initialize detector
    detector = Detector(
        model_path="yolov8n.pt",  # Use nano for demo
        confidence_threshold=0.35,
        device="cpu"  # Use CPU for demo
    )
    
    # Run detection
    detections = detector.detect(frame, frame_id=0)
    
    print(f"Detected {detections.num_detections} persons")
    
    # Visualize
    annotator = VideoAnnotator()
    annotated = annotator.annotate_frame(
        frame,
        detections.raw_boxes,
        list(range(detections.num_detections)),
        detections.confidences
    )
    
    # Display
    plt.figure(figsize=(12, 8))
    plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    plt.title(f'Detected {detections.num_detections} persons')
    plt.axis('off')
    plt.show()
    
    return detections

# Create a test image
print("Detection demo - provide an image path to test")
# detections = demo_detection("path/to/image.jpg")

## 6. Configuration Explorer

In [ ]:
# Interactive configuration explorer
def create_config_ui():
    model_dropdown = widgets.Dropdown(
        options=['yolov8n.pt', 'yolov8s.pt', 'yolov8m.pt', 'yolov8l.pt', 'yolov8x.pt'],
        value='yolov8x.pt',
        description='Model:'
    )
    
    conf_slider = widgets.FloatSlider(
        value=0.35,
        min=0.1,
        max=0.9,
        step=0.05,
        description='Confidence:'
    )
    
    tracker_dropdown = widgets.Dropdown(
        options=['bytetrack', 'botsort'],
        value='bytetrack',
        description='Tracker:'
    )
    
    ensemble_checkbox = widgets.Checkbox(
        value=True,
        description='Ensemble Tracking'
    )
    
    reid_checkbox = widgets.Checkbox(
        value=True,
        description='Enable ReID'
    )
    
    def on_change(change):
        config_display.value = f"""
**Current Configuration:**
- Model: {model_dropdown.value}
- Confidence: {conf_slider.value}
- Primary Tracker: {tracker_dropdown.value}
- Ensemble: {ensemble_checkbox.value}
- ReID: {reid_checkbox.value}
        """
    
    for widget in [model_dropdown, conf_slider, tracker_dropdown, ensemble_checkbox, reid_checkbox]:
        widget.observe(on_change, names='value')
    
    config_display = widgets.HTML(value="")
    on_change(None)
    
    return widgets.VBox([
        widgets.HTML('<h3>🔧 Pipeline Configuration</h3>'),
        model_dropdown,
        conf_slider,
        tracker_dropdown,
        ensemble_checkbox,
        reid_checkbox,
        config_display
    ])

config_ui = create_config_ui()
display(config_ui)

## 7. Cleanup

In [ ]:
# Clean up resources
# if 'viewer' in dir():
#     viewer.close()
# if 'pipeline' in dir():
#     pipeline.cleanup()

print("✅ Demo complete!")